# 3. Preparación de los Datos

Fase 3 de CRISP-DM: limpieza, pre-procesamiento y construcción de nuevos datos (feature engineering) para el modelado.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.feature_selection import mutual_info_regression, SelectKBest

DATA_DIR = "../data"
CLEAN_CSV = f"{DATA_DIR}/processed/gb_videos_clean.csv"
MODEL_CSV = f"{DATA_DIR}/processed/gb_videos_model.csv"

In [ ]:
# Cargamos el dataset limpio generado en el notebook 04
df = pd.read_csv(CLEAN_CSV, parse_dates=["publish_time", "trending_date"])
print("Shape:", df.shape)

## 3.1 Variables derivadas (construir nuevos datos)

In [ ]:
# Construcción de nuevas variables (feature engineering)
cols_antes = set(df.columns)

# Temporales
df["publish_month"] = df["publish_time"].dt.month
df["is_weekend"] = df["publish_time"].dt.weekday.isin([5, 6]).astype(int)
df["days_to_trending"] = (df["trending_date"] - df["publish_time"]).dt.days.clip(lower=0)

# Texto
df["title_length"] = df["title"].astype(str).str.len()
df["description_length"] = df["description"].astype(str).str.len()
df["tags_count"] = df["tags"].astype(str).str.split("|").str.len()

# Interacción / viralidad
df["dislikes_per_view"] = df["dislikes"] / (df["views"] + 1)
df["interaction_rate"] = (df["likes"] + df["comment_count"]) / (df["views"] + 1)
df["views_per_day"] = df["views"] / (df["days_to_trending"] + 1)

nuevas = [c for c in df.columns if c not in cols_antes]
print("Nuevas columnas:", nuevas)

In [ ]:
display(df[['days_to_trending','engagement_rate','like_dislike_ratio','title_length','tags_count']].describe().T)

## 3.2 Encoding y escalado (pre-procesar datos)

In [ ]:
# Encoding de categóricas (one-hot, limitado a top 20) + escalado de numéricas
target = "views"

# One-hot de category_name y state (agrupando poco frecuentes en 'Other')
df_enc = df.copy()
for col in ["category_name", "state"]:
    top = df_enc[col].value_counts().head(20).index
    df_enc[col] = df_enc[col].where(df_enc[col].isin(top), "Other")

encoder = OneHotEncoder(sparse_output=False, drop="first", handle_unknown="ignore")
encoded = encoder.fit_transform(df_enc[["category_name", "state"]])
encoded_df = pd.DataFrame(encoded, columns=encoder.get_feature_names_out(["category_name", "state"]), index=df_enc.index)

# Variables numéricas predictoras
numeric_feats = [
    "likes", "dislikes", "comment_count", "engagement_rate", "likes_per_view",
    "dislikes_per_view", "comments_per_view", "like_dislike_ratio", "interaction_rate",
    "days_to_trending", "views_per_day", "title_length", "description_length",
    "tags_count", "publish_month", "publish_hour", "publish_weekday", "is_weekend",
]
numeric_feats = [c for c in numeric_feats if c in df_enc.columns]

model_df = pd.concat([df_enc[numeric_feats + [target]], encoded_df], axis=1).dropna()

# Escalado de las numéricas
scaler = StandardScaler()
model_df[numeric_feats] = scaler.fit_transform(model_df[numeric_feats])

print("Shape modelado:", model_df.shape)
print("Ejemplo de variables:", numeric_feats[:8], "...")

## 3.3 Selección de variables

In [ ]:
# Selección de variables por información mutua respecto a 'views'
X = model_df.drop(columns=target)
y = model_df[target]

selector = SelectKBest(score_func=mutual_info_regression, k=min(15, X.shape[1]))
selector.fit(X, y)
scores = pd.Series(selector.scores_, index=X.columns).sort_values(ascending=False)
display(scores.head(15).to_frame("mutual_info"))

## 3.4 Guardar dataset para modelado

In [ ]:
# Guardar dataset listo para modelado
model_df.to_csv(MODEL_CSV, index=False)
print("Dataset de modelado guardado en", MODEL_CSV)